# TP Chicago Crimes — Pandas, Spark, HDFS et API

**Objectif :** traiter l’ensemble des questions du TP à partir du jeu de données officiel  
**Crimes — 2001 to Present** de la Ville de Chicago.

> Le jeu de données est mis à jour quotidiennement. Le nombre de lignes, la taille du fichier,
> les temps d’exécution et le nombre de tâches Spark doivent donc être **mesurés lors de
> l’exécution** et ne doivent pas être codés en dur.

## Conseils d’exécution

1. Attacher ce notebook à un cluster Databricks suffisamment dimensionné.
2. Exécuter les cellules dans l’ordre.
3. Passer `DOWNLOAD_FULL_CSV` à `true` lors du premier lancement.
4. Les cellules les plus coûteuses sont protégées par des paramètres.
5. Conserver les sorties et captures utiles avant de rendre le notebook.

**Nom / Prénom :** Terosier Eddy
**Date :** 28/07/2026

## 0. Configuration du notebook

Ce notebook est configuré pour **Databricks Serverless** et utilise directement le fichier
déjà importé dans Unity Catalog :

`/Volumes/workspace/default/tp-chicagoo/Crimes_-_2001_to_Present_trimmed.csv`

Aucun téléchargement Internet automatique n'est nécessaire.


In [0]:
# CONFIGURATION SERVERLESS

BASE_PATH = "/Volumes/workspace/default/tp-chicagoo"
CSV_PATH = (
    "/Volumes/workspace/default/tp-chicagoo/"
    "Crimes_-_2001_to_Present_trimmed.csv"
)
LOCAL_CSV_PATH = CSV_PATH

RUN_PANDAS_FULL = False
RUN_LONG_STATS = False
RUN_EXACT_UNIQUES = False

RUN_HDFS_WRITE = False
HDFS_TARGET = "hdfs://namenode:9000/tp/chicago_crimes"

RUN_API_DOWNLOAD = False
API_YEAR = 2024
SOCRATA_APP_TOKEN = ""

print("BASE_PATH             =", BASE_PATH)
print("CSV_PATH              =", CSV_PATH)
print("RUN_PANDAS_FULL       =", RUN_PANDAS_FULL)
print("RUN_LONG_STATS        =", RUN_LONG_STATS)
print("RUN_EXACT_UNIQUES     =", RUN_EXACT_UNIQUES)
print("RUN_HDFS_WRITE        =", RUN_HDFS_WRITE)
print("RUN_API_DOWNLOAD      =", RUN_API_DOWNLOAD)
print("API_YEAR              =", API_YEAR)


BASE_PATH             = /Volumes/workspace/default/tp-chicagoo
CSV_PATH              = /Volumes/workspace/default/tp-chicagoo/Crimes_-_2001_to_Present_trimmed.csv
RUN_PANDAS_FULL       = False
RUN_LONG_STATS        = False
RUN_EXACT_UNIQUES     = False
RUN_HDFS_WRITE        = False
RUN_API_DOWNLOAD      = False
API_YEAR              = 2024


In [0]:
# Imports
import gc
import json
import os
import time
from pathlib import Path
from typing import Any, Callable, Dict, List, Tuple

import pandas as pd
import requests

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

print("Version pandas :", pd.__version__)
print("Version Spark  :", spark.version)
print("Mode de calcul : Databricks Serverless")


Version pandas : 2.2.3
Version Spark  : 4.1.0
Mode de calcul : Databricks Serverless


### Fonctions utilitaires

Le mode Serverless interdit certaines API bas niveau (`SparkContext`, `StatusTracker`, `.rdd`).
Le notebook utilise donc uniquement les API DataFrame compatibles.


In [0]:
def elapsed_call(label: str, fn: Callable[[], Any]) -> Tuple[Any, float]:
    start = time.perf_counter()
    result = fn()
    elapsed = time.perf_counter() - start
    print(f"{label}: {elapsed:.2f} s")
    return result, elapsed


def serverless_partition_count(df: DataFrame) -> int:
    return (
        df.select(F.spark_partition_id().alias("partition_id"))
        .distinct()
        .count()
    )


def storage_path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def human_bytes(value: int) -> str:
    units = ["o", "Ko", "Mo", "Go", "To"]
    size = float(value)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
    return f"{size:.2f} To"


# 1. Vérifier les données

Le fichier a déjà été téléchargé puis importé dans Unity Catalog.
La version utilisée est une version réduite du fichier officiel.


In [0]:
print("Métadonnées en ligne non interrogées : accès Internet indisponible sur ce compute.")
print("Fichier utilisé :", CSV_PATH)


Métadonnées en ligne non interrogées : accès Internet indisponible sur ce compute.
Fichier utilisé : /Volumes/workspace/default/tp-chicagoo/Crimes_-_2001_to_Present_trimmed.csv


In [0]:
print("Aucun téléchargement automatique n'est lancé.")
print("Le CSV est déjà présent dans le volume Unity Catalog.")


Aucun téléchargement automatique n'est lancé.
Le CSV est déjà présent dans le volume Unity Catalog.


In [0]:
# Vérification du fichier et aperçu brut
display(dbutils.fs.ls(BASE_PATH))
print(dbutils.fs.head(CSV_PATH, 2_000))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/tp-chicagoo/Crimes_-_2001_to_Present_trimmed.csv,Crimes_-_2001_to_Present_trimmed.csv,523239190,1785233632000


[Truncated to first 2000 bytes]
ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,true,false,1033,010,25,30,17,,,2022,04/18/2024 03:40:59 PM,,,
13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,true,false,1122,011,28,26,18,,,2023,01/20/2024 03:41:12 PM,,,
11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,false,false,0122,001,42,32,02,,,2017,02/11/2018 03:57:41 PM,,,
12416972,JE293535,10/01/2020 12:01:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,RESIDENCE,false,true,0923,009,14,63,02,,,2020,09/14/2023 03:41:59 PM,,,
12536164,JE439378,09/24/

# 2. Lecture avec Pandas et mesure du temps

Pandas charge le DataFrame dans la mémoire du **driver**. Une lecture complète peut provoquer
un ralentissement important ou une erreur de mémoire selon la taille du driver.

La cellule commence par mesurer un échantillon de 100 000 lignes. La lecture complète n’est
effectuée que si `RUN_PANDAS_FULL=true`.

In [0]:
sample_rows = 100_000

pandas_sample, pandas_sample_seconds = elapsed_call(
    f"Lecture pandas de {sample_rows:,} lignes",
    lambda: pd.read_csv(
        LOCAL_CSV_PATH,
        nrows=sample_rows,
        low_memory=False,
    ),
)

sample_memory = int(
    pandas_sample.memory_usage(index=True, deep=True).sum()
)

print("Dimensions de l'échantillon :", pandas_sample.shape)
print("Mémoire profonde de l'échantillon :", human_bytes(sample_memory))
display(pandas_sample.head())


Lecture pandas de 100,000 lignes: 0.77 s
Dimensions de l'échantillon : (100000, 22)
Mémoire profonde de l'échantillon : 66.08 Mo


ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,true,false,1033,10,25.0,30.0,17,null,null,2022,04/18/2024 03:40:59 PM,null,null,null
13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,true,false,1122,11,28.0,26.0,18,null,null,2023,01/20/2024 03:41:12 PM,null,null,null
11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,false,false,122,1,42.0,32.0,02,null,null,2017,02/11/2018 03:57:41 PM,null,null,null
12416972,JE293535,10/01/2020 12:01:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,RESIDENCE,false,true,923,9,14.0,63.0,02,null,null,2020,09/14/2023 03:41:59 PM,null,null,null
12536164,JE439378,09/24/2015 12:00:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,923,9,14.0,63.0,02,null,null,2015,09/14/2023 03:41:59 PM,null,null,null


In [0]:
pandas_full_seconds = None
pandas_full_memory = None
pandas_full_shape = None
pandas_full_error = None

if RUN_PANDAS_FULL:
    try:
        pandas_full, pandas_full_seconds = elapsed_call(
            "Lecture pandas complète",
            lambda: pd.read_csv(LOCAL_CSV_PATH, low_memory=False),
        )
        pandas_full_shape = pandas_full.shape
        pandas_full_memory = int(
            pandas_full.memory_usage(index=True, deep=True).sum()
        )

        print("Dimensions :", pandas_full_shape)
        print("Mémoire profonde :", human_bytes(pandas_full_memory))
        display(pandas_full.head())
    except (MemoryError, Exception) as exc:
        pandas_full_error = repr(exc)
        print("La lecture pandas complète a échoué :", pandas_full_error)
else:
    print(
        "Lecture complète désactivée. Passer RUN_PANDAS_FULL à true "
        "pour obtenir le temps réel sur ce cluster."
    )

Lecture pandas complète: 11.54 s
Dimensions : (2214609, 22)
Mémoire profonde : 1.47 Go


ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,true,false,1033,10.0,25.0,30.0,17,null,null,2022,04/18/2024 03:40:59 PM,null,null,null
13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,true,false,1122,11.0,28.0,26.0,18,null,null,2023,01/20/2024 03:41:12 PM,null,null,null
11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,false,false,122,1.0,42.0,32.0,02,null,null,2017,02/11/2018 03:57:41 PM,null,null,null
12416972,JE293535,10/01/2020 12:01:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,RESIDENCE,false,true,923,9.0,14.0,63.0,02,null,null,2020,09/14/2023 03:41:59 PM,null,null,null
12536164,JE439378,09/24/2015 12:00:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,923,9.0,14.0,63.0,02,null,null,2015,09/14/2023 03:41:59 PM,null,null,null


### Conclusion Q2

- Pandas exécute le traitement sur une seule machine : le driver.
- Le fichier CSV brut et le DataFrame Pandas coexistent potentiellement en mémoire ou sur le
  disque local.
- Les colonnes de type texte peuvent consommer beaucoup plus de mémoire que leur taille brute.
- Sur un jeu de plusieurs millions de lignes, Pandas peut devenir très coûteux en temps et en
  mémoire, voire provoquer une erreur `OutOfMemory`.
- Spark permet de répartir le travail entre plusieurs partitions et plusieurs exécuteurs.

Il ne faut toutefois pas conclure que Spark sera automatiquement plus rapide sur n’importe
quel cluster : le démarrage des tâches et les échanges distribués ont un coût. Son avantage
principal est la **scalabilité** et l’absence de collecte automatique de toutes les données
dans la mémoire du driver.

# 3. Lecture avec Spark

## 3.1 Sans option, puis avec `header=True`

La durée de construction du DataFrame et celle de l’action `count()` sont séparées pour
illustrer l’évaluation paresseuse de Spark.

In [0]:
# Lecture avec les options par défaut
df_default, default_build_seconds = elapsed_call(
    "Construction du DataFrame sans option",
    lambda: spark.read.csv(CSV_PATH),
)

default_count, default_count_seconds = elapsed_call(
    "Count sans header",
    df_default.count,
)

print("Nombre de lignes sans header :", default_count)
print("Colonnes générées :", df_default.columns)
df_default.show(5, truncate=False)


Construction du DataFrame sans option: 0.00 s
Count sans header: 1.62 s
Nombre de lignes sans header : 2214610
Colonnes générées : ['_c0', '_c1', '_c2', '_c3', '_c4', '_c5', '_c6', '_c7', '_c8', '_c9', '_c10', '_c11', '_c12', '_c13', '_c14', '_c15', '_c16', '_c17', '_c18', '_c19', '_c20', '_c21']
+--------+-----------+----------------------+-----------------------+----+--------------------------+----------------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+--------+---------+--------+
|_c0     |_c1        |_c2                   |_c3                    |_c4 |_c5                       |_c6                                     |_c7                 |_c8   |_c9     |_c10|_c11    |_c12|_c13          |_c14    |_c15        |_c16        |_c17|_c18                  |_c19    |_c20     |_c21    |
+--------+-----------+----------------------+-----------------------+----+------------------

In [0]:
# Lecture avec la première ligne utilisée comme en-tête
df_header, header_build_seconds = elapsed_call(
    "Construction du DataFrame avec header=True",
    lambda: spark.read.option("header", "true").csv(CSV_PATH),
)

header_count, header_count_seconds = elapsed_call(
    "Count avec header",
    df_header.count,
)

print("Nombre de lignes de données :", header_count)
print("Nombre de colonnes :", len(df_header.columns))
print("Noms des colonnes :", df_header.columns)
display(df_header.limit(5))


Construction du DataFrame avec header=True: 0.00 s
Count avec header: 1.56 s
Nombre de lignes de données : 2214609
Nombre de colonnes : 22
Noms des colonnes : ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,true,false,1033,010,25,30,17,null,null,2022,04/18/2024 03:40:59 PM,null,null,null
13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,true,false,1122,011,28,26,18,null,null,2023,01/20/2024 03:41:12 PM,null,null,null
11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,false,false,0122,001,42,32,02,null,null,2017,02/11/2018 03:57:41 PM,null,null,null
12416972,JE293535,10/01/2020 12:01:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,RESIDENCE,false,true,0923,009,14,63,02,null,null,2020,09/14/2023 03:41:59 PM,null,null,null
12536164,JE439378,09/24/2015 12:00:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,0923,009,14,63,02,null,null,2015,09/14/2023 03:41:59 PM,null,null,null


In [0]:
comparison_rows = [
    ("Spark sans header", default_build_seconds, default_count_seconds, default_count),
    ("Spark avec header", header_build_seconds, header_count_seconds, header_count),
]

display(
    spark.createDataFrame(
        comparison_rows,
        ["lecture", "construction_secondes", "count_secondes", "lignes"],
    )
)


lecture,construction_secondes,count_secondes,lignes
Spark sans header,6.372900020323868E-5,1.6186064159999205,2214610
Spark avec header,6.971900006647047E-5,1.5647779120001815,2214609


### Interprétation

- Sans `header=True`, la ligne d’en-tête est traitée comme une ligne de données et les colonnes
  portent les noms `_c0`, `_c1`, etc.
- Avec `header=True`, Spark récupère les vrais noms des colonnes et ne compte plus l’en-tête
  comme une observation.
- Le temps significatif est surtout celui de l’action `count()`, car Spark évalue les
  transformations paresseusement.

## 3.2 Nombre de partitions, colonnes et lignes

In [0]:
partition_count, partition_seconds = elapsed_call(
    "Calcul du nombre de partitions",
    lambda: serverless_partition_count(df_header),
)

column_names = df_header.columns
row_count = header_count

print("Nombre de partitions :", partition_count)
print("Nombre de colonnes   :", len(column_names))
print("Nombre de lignes     :", row_count)
print("Colonnes              :", column_names)


Calcul du nombre de partitions: 0.99 s
Nombre de partitions : 8
Nombre de colonnes   : 22
Nombre de lignes     : 2214609
Colonnes              : ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


# 4. Affichage agréable dans Databricks

Dans Databricks, `display(df)` produit une table interactive. Pour un comportement proche
d’un notebook Pandas dans d’autres environnements PySpark, on peut aussi activer
`spark.sql.repl.eagerEval.enabled`.

In [0]:
# Affichage  compatible Databricks Serverless

display(df_header.limit(20))

ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
13311263,JG503434,07/29/2022 03:39:00 AM,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,true,false,1033,010,25,30,17,null,null,2022,04/18/2024 03:40:59 PM,null,null,null
13053066,JG103252,01/03/2023 04:44:00 PM,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,true,false,1122,011,28,26,18,null,null,2023,01/20/2024 03:41:12 PM,null,null,null
11227634,JB147599,08/26/2017 10:00:00 AM,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,false,false,0122,001,42,32,02,null,null,2017,02/11/2018 03:57:41 PM,null,null,null
12416972,JE293535,10/01/2020 12:01:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,RESIDENCE,false,true,0923,009,14,63,02,null,null,2020,09/14/2023 03:41:59 PM,null,null,null
12536164,JE439378,09/24/2015 12:00:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,0923,009,14,63,02,null,null,2015,09/14/2023 03:41:59 PM,null,null,null
12536166,JE439332,09/07/2014 12:00:00 AM,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,0923,009,14,63,02,null,null,2014,09/14/2023 03:41:59 PM,null,null,null
12829020,JF398208,01/01/2015 12:01:00 AM,080XX S WOOD ST,0266,CRIMINAL SEXUAL ASSAULT,PREDATORY,RESIDENCE,false,true,0611,006,17,71,02,null,null,2015,09/14/2023 03:41:59 PM,null,null,null
12888104,JF469015,11/10/2022 03:47:00 AM,072XX S MAY ST,1477,WEAPONS VIOLATION,RECKLESS FIREARM DISCHARGE,STREET,false,false,0733,007,17,68,15,1169903,1856822,2022,09/14/2023 03:41:59 PM,41.76261474,-87.652840463,"(41.76261474, -87.652840463)"
12914647,JF500216,07/10/2013 12:00:00 AM,046XX S SAWYER AVE,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,0821,008,12,58,02,null,null,2013,09/14/2023 03:41:59 PM,null,null,null
12978205,JG146937,08/01/2022 10:00:00 PM,028XX S KEELER AVE,1562,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,APARTMENT,false,true,1031,010,22,30,17,null,null,2022,09/14/2023 03:41:59 PM,null,null,null


# 5. Lecture avec `inferSchema=True`

Sans schéma explicite, Spark doit analyser les valeurs afin de déterminer les types. Cette
analyse implique une lecture supplémentaire du CSV. Sur plusieurs millions de lignes, cela
peut prendre des dizaines de secondes ou davantage selon le cluster.

In [1]:
df_inferred, infer_build_seconds = elapsed_call(
    "Construction avec header=True et inferSchema=True",
    lambda: (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(CSV_PATH)
    ),
)

inferred_count, infer_count_seconds = elapsed_call(
    "Count après inférence du schéma",
    df_inferred.count,
)

print("Temps de construction avec inférence :", round(infer_build_seconds, 2), "s")
print("Temps du count après inférence        :", round(infer_count_seconds, 2), "s")
print("Nombre de lignes                      :", inferred_count)
df_inferred.printSchema()


NameError: name 'elapsed_call' is not defined

### Pourquoi le chargement est-il plus long ?

Pour inférer le schéma d’un CSV, Spark doit examiner les données afin de décider si chaque
colonne correspond à un entier, un flottant, un booléen, une date ou une chaîne. Cette passe
s’ajoute ensuite au traitement demandé par l’action suivante. Il est donc préférable de
définir manuellement le schéma lorsque la structure est connue.

# 6. Types, accès aux colonnes et transformations

In [0]:
# Types résultant de inferSchema
inferred_types = [(field.name, field.dataType.simpleString()) for field in df_inferred.schema.fields]
display(spark.createDataFrame(inferred_types, ["colonne", "type_spark"]))

colonne,type_spark
ID,int
Case Number,string
Date,string
Block,string
IUCR,string
Primary Type,string
Description,string
Location Description,string
Arrest,boolean
Domestic,boolean


In [0]:
# Plusieurs méthodes d'accès aux colonnes
district_column_1 = df_inferred["District"]
district_column_2 = F.col("District")

# Pour un nom contenant un espace, F.col ou df["..."] est recommandé.
case_number_column_1 = df_inferred["Case Number"]
case_number_column_2 = F.col("Case Number")

# DataFrame contenant uniquement District et Case Number
df_district_case = df_inferred.select(
    F.col("District"),
    F.col("Case Number"),
)
display(df_district_case.limit(10))

District,Case Number
10,JG503434
11,JG103252
1,JB147599
9,JE293535
9,JE439378
9,JE439332
6,JF398208
7,JF469015
8,JF500216
10,JG146937


In [0]:
# Ajouter une colonne constante égale à 1
df_with_one = df_district_case.withColumn("one", F.lit(1))
display(df_with_one.limit(5))

# Supprimer cette colonne
df_without_one = df_with_one.drop("one")
display(df_without_one.limit(5))

# Supprimer ID, Case Number et Block du DataFrame d'origine
df_without_id_case_block = df_inferred.drop("ID", "Case Number", "Block")
print(df_without_id_case_block.columns)

District,Case Number,one
10,JG503434,1
11,JG103252,1
1,JB147599,1
9,JE293535,1
9,JE439378,1


District,Case Number
10,JG503434
11,JG103252
1,JB147599
9,JE293535
9,JE439378


['Date', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


# 7. Affichage de colonnes et mode vertical

In [0]:
# Les cinq premières observations de IUCR
df_inferred.select("IUCR").show(5, truncate=False)

# Les cinq premières observations de Case Number, Date et Arrest
df_inferred.select("Case Number", "Date", "Arrest").show(5, truncate=False)

# Deux enregistrements en mode vertical, sans troncature
df_inferred.show(n=2, truncate=False, vertical=True)

+----+
|IUCR|
+----+
|1582|
|2017|
|0281|
|1753|
|1753|
+----+
only showing top 5 rows
+-----------+----------------------+------+
|Case Number|Date                  |Arrest|
+-----------+----------------------+------+
|JG503434   |07/29/2022 03:39:00 AM|true  |
|JG103252   |01/03/2023 04:44:00 PM|true  |
|JB147599   |08/26/2017 10:00:00 AM|false |
|JE293535   |10/01/2020 12:01:00 AM|false |
|JE439378   |09/24/2015 12:00:00 AM|false |
+-----------+----------------------+------+
only showing top 5 rows
-RECORD 0---------------------------------------------
 ID                   | 13311263                      
 Case Number          | JG503434                      
 Date                 | 07/29/2022 03:39:00 AM        
 Block                | 023XX S TROY ST               
 IUCR                 | 1582                          
 Primary Type         | OFFENSE INVOLVING CHILDREN    
 Description          | CHILD PORNOGRAPHY             
 Location Description | RESIDENCE                    

# 8. Définition manuelle du schéma

Le CSV officiel contient actuellement 22 colonnes. Les identifiants codifiés tels que
`IUCR` et `FBI Code` restent des chaînes afin de préserver les zéros initiaux et les valeurs
alphanumériques.

Les champs `Date` et `Updated On` sont lus comme des timestamps au format utilisé par
l’export CSV Socrata.

In [0]:
crime_schema = StructType([
    StructField("ID", LongType(), True),
    StructField("Case Number", StringType(), True),
    StructField("Date", TimestampType(), True),
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community Area", IntegerType(), True),
    StructField("FBI Code", StringType(), True),
    StructField("X Coordinate", IntegerType(), True),
    StructField("Y Coordinate", IntegerType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated On", TimestampType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True),
])

df, manual_schema_build_seconds = elapsed_call(
    "Construction avec schéma manuel",
    lambda: (
        spark.read
        .option("header", "true")
        .option("timestampFormat", "MM/dd/yyyy hh:mm:ss a")
        .option("locale", "en-US")
        .schema(crime_schema)
        .csv(CSV_PATH)
    ),
)

manual_count, manual_count_seconds = elapsed_call(
    "Count avec schéma manuel",
    df.count,
)

df.printSchema()
print("Nombre de lignes :", manual_count)
display(df.limit(10))


Construction avec schéma manuel: 0.00 s
Count avec schéma manuel: 1.25 s
root
 |-- ID: long (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: integer (nullable = true)
 |-- District: integer (nullable = true)
 |-- Ward: integer (nullable = true)
 |-- Community Area: integer (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: integer (nullable = true)
 |-- Y Coordinate: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Updated On: timestamp (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)

Nombre de lignes : 2

ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
13311263,JG503434,2022-07-29T03:39:00.000Z,023XX S TROY ST,1582,OFFENSE INVOLVING CHILDREN,CHILD PORNOGRAPHY,RESIDENCE,true,false,1033,10,25,30,17,null,null,2022,2024-04-18T15:40:59.000Z,null,null,null
13053066,JG103252,2023-01-03T16:44:00.000Z,039XX W WASHINGTON BLVD,2017,NARCOTICS,MANUFACTURE / DELIVER - CRACK,SIDEWALK,true,false,1122,11,28,26,18,null,null,2023,2024-01-20T15:41:12.000Z,null,null,null
11227634,JB147599,2017-08-26T10:00:00.000Z,001XX W RANDOLPH ST,0281,CRIM SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL/MOTEL,false,false,122,1,42,32,02,null,null,2017,2018-02-11T15:57:41.000Z,null,null,null
12416972,JE293535,2020-10-01T00:01:00.000Z,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,RESIDENCE,false,true,923,9,14,63,02,null,null,2020,2023-09-14T15:41:59.000Z,null,null,null
12536164,JE439378,2015-09-24T00:00:00.000Z,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,923,9,14,63,02,null,null,2015,2023-09-14T15:41:59.000Z,null,null,null
12536166,JE439332,2014-09-07T00:00:00.000Z,031XX W 53RD PL,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,923,9,14,63,02,null,null,2014,2023-09-14T15:41:59.000Z,null,null,null
12829020,JF398208,2015-01-01T00:01:00.000Z,080XX S WOOD ST,0266,CRIMINAL SEXUAL ASSAULT,PREDATORY,RESIDENCE,false,true,611,6,17,71,02,null,null,2015,2023-09-14T15:41:59.000Z,null,null,null
12888104,JF469015,2022-11-10T03:47:00.000Z,072XX S MAY ST,1477,WEAPONS VIOLATION,RECKLESS FIREARM DISCHARGE,STREET,false,false,733,7,17,68,15,1169903,1856822,2022,2023-09-14T15:41:59.000Z,41.76261474,-87.652840463,"(41.76261474, -87.652840463)"
12914647,JF500216,2013-07-10T00:00:00.000Z,046XX S SAWYER AVE,1753,OFFENSE INVOLVING CHILDREN,SEXUAL ASSAULT OF CHILD BY FAMILY MEMBER,APARTMENT,false,true,821,8,12,58,02,null,null,2013,2023-09-14T15:41:59.000Z,null,null,null
12978205,JG146937,2022-08-01T22:00:00.000Z,028XX S KEELER AVE,1562,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,APARTMENT,false,true,1031,10,22,30,17,null,null,2022,2023-09-14T15:41:59.000Z,null,null,null


In [0]:
# Contrôle du parsing des colonnes essentielles
parsing_control = df.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("ID").isNull().cast("long")).alias("id_nulls"),
    F.sum(F.col("Date").isNull().cast("long")).alias("date_nulls"),
    F.sum(F.col("Updated On").isNull().cast("long")).alias("updated_on_nulls"),
    F.sum(F.col("Arrest").isNull().cast("long")).alias("arrest_nulls"),
).first()

print(parsing_control)

if parsing_control["date_nulls"] == parsing_control["rows"]:
    print(
        "ATTENTION : toutes les dates sont nulles. Vérifier le format réel du CSV "
        "avec dbutils.fs.head(CSV_PATH), puis adapter timestampFormat."
    )

Row(rows=2214609, id_nulls=0, date_nulls=0, updated_on_nulls=0, arrest_nulls=0)


# 9. `limit()`, `head()`, `take()` et `collect()`

| Méthode | Résultat | Évaluation |
|---|---|---|
| `limit(n)` | Nouveau DataFrame limité à `n` lignes | Transformation paresseuse |
| `head()` | Une `Row` | Action |
| `head(n)` | Liste de `n` objets `Row` | Action |
| `take(n)` | Liste de `n` objets `Row` | Action |
| `collect()` | Toutes les lignes dans une liste Python | Action très dangereuse sur un gros DataFrame |

`df.limit()` sans argument est invalide : le nombre de lignes doit être fourni.

In [0]:
# limit nécessite un argument
limited_df = df.limit(5)
print("Type retourné par limit(5) :", type(limited_df))
limited_df.show(truncate=False)

# head()
first_row = df.head()
print("df.head() :", first_row)

# head(n)
first_three_head = df.head(3)
print("df.head(3) :", first_three_head)

# take(n)
first_three_take = df.take(3)
print("df.take(3) :", first_three_take)

Type retourné par limit(5) : <class 'pyspark.sql.connect.dataframe.DataFrame'>
+--------+-----------+-------------------+-----------------------+----+--------------------------+----------------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+--------+---------+--------+
|ID      |Case Number|Date               |Block                  |IUCR|Primary Type              |Description                             |Location Description|Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On         |Latitude|Longitude|Location|
+--------+-----------+-------------------+-----------------------+----+--------------------------+----------------------------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+-------------------+--------+---------+--------+
|13311263|JG503434   |20

In [0]:
# NE PAS exécuter df.collect() sur plus de 8 millions de lignes :
# all_rows = df.collect()

# Démonstration sûre sur un sous-ensemble
safe_collect = df.limit(100).collect()
print("Nombre d'objets Row collectés de façon sûre :", len(safe_collect))
print(safe_collect[:2])

Nombre d'objets Row collectés de façon sûre : 100
[Row(ID=13311263, Case Number='JG503434', Date=datetime.datetime(2022, 7, 29, 3, 39), Block='023XX S TROY ST', IUCR='1582', Primary Type='OFFENSE INVOLVING CHILDREN', Description='CHILD PORNOGRAPHY', Location Description='RESIDENCE', Arrest=True, Domestic=False, Beat=1033, District=10, Ward=25, Community Area=30, FBI Code='17', X Coordinate=None, Y Coordinate=None, Year=2022, Updated On=datetime.datetime(2024, 4, 18, 15, 40, 59), Latitude=None, Longitude=None, Location=None), Row(ID=13053066, Case Number='JG103252', Date=datetime.datetime(2023, 1, 3, 16, 44), Block='039XX W WASHINGTON BLVD', IUCR='2017', Primary Type='NARCOTICS', Description='MANUFACTURE / DELIVER - CRACK', Location Description='SIDEWALK', Arrest=True, Domestic=False, Beat=1122, District=11, Ward=28, Community Area=26, FBI Code='18', X Coordinate=None, Y Coordinate=None, Year=2023, Updated On=datetime.datetime(2024, 1, 20, 15, 41, 12), Latitude=None, Longitude=None, Loc

### Conclusion Q9

`collect()` transfère l’intégralité des partitions vers la mémoire du driver. Sur ce jeu de
données, cette opération peut saturer le driver et faire échouer le notebook. Il faut préférer
`limit`, `take`, `head`, des agrégations Spark ou l’écriture distribuée.

# 10. Mise en cache du DataFrame

`cache()` utilise par défaut le niveau `MEMORY_AND_DISK_DESER`. La mise en cache est
paresseuse : une action comme `count()` est nécessaire pour la matérialiser.

In [0]:
df = (
    spark.read
    .option("header", "true")
    .option("timestampFormat", "MM/dd/yyyy hh:mm:ss a")
    .option("locale", "en-US")
    .schema(crime_schema)
    .csv(CSV_PATH)
)

In [0]:
#MISE EN CACHE
# le cache() / persist() n'est pas supporté sur ce compute Serverless.
# je garde donc simplement une référence vers le DataFrame d'origine.

df_cached = df

print(
    "Cache non exécuté : cache/persist n'est pas supporté "
    "sur Databricks Serverless."
)

print("Le notebook continue avec df_cached = df.")


Cache non exécuté : cache/persist n'est pas supporté sur Databricks Serverless.
Le notebook continue avec df_cached = df.


# 11. `describe()` versus `summary()`

- `describe()` retourne `count`, `mean`, `stddev`, `min` et `max`.
- `summary()` ajoute notamment les quantiles `25%`, `50%` et `75%`.

Sur Serverless, les API bas niveau utilisées pour compter automatiquement les tâches ne sont
pas disponibles. Le nombre de tâches doit être relevé dans **See performance** sous la cellule.


In [0]:
if RUN_LONG_STATS:
    describe_df = df_cached.describe()
    describe_rows, describe_seconds = elapsed_call(
        "df.describe().collect()",
        describe_df.collect,
    )
    display(spark.createDataFrame(describe_rows, describe_df.schema))

    summary_df = df_cached.summary()
    summary_rows, summary_seconds = elapsed_call(
        "df.summary().collect()",
        summary_df.collect,
    )
    display(spark.createDataFrame(summary_rows, summary_df.schema))

    print("Temps describe() :", describe_seconds, "secondes")
    print("Temps summary()  :", summary_seconds, "secondes")
    print("Nombre de tâches : ouvrir 'See performance' sous cette cellule.")
else:
    print(
        "Mettre RUN_LONG_STATS = True dans la configuration, "
        "puis relancer cette cellule."
    )

    demo_stats_df = df_cached.select(
        "ID", "Beat", "District", "Ward", "Community Area",
        "X Coordinate", "Y Coordinate", "Year", "Latitude", "Longitude"
    ).limit(100_000)

    display(demo_stats_df.describe())
    display(demo_stats_df.summary())


df.describe().collect(): 2.94 s


summary,ID,Case Number,Block,IUCR,Primary Type,Description,Location Description,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Latitude,Longitude,Location
count,2214609,2214609,2214609,2214609,2214609,2214609,2202515,2214609,2214608,2196737,2196632,2214609,2117542,2117542,2214609,2117542,2117542,2117542
mean,1.1291700307641665E7,null,null,1086.5438042985438,null,null,null,1148.8881472982364,11.237249210695527,23.159755127718977,36.843304659132706,11.134632881263046,1164941.9766181733,1886275.485424138,2018.0816663347796,41.84352838029658,-87.67022922969481,null
stddev,1778633.200589106,null,null,820.9559413709741,null,null,null,698.7379273929268,6.97672373548794,13.984877062000939,21.490912193034593,6.788762659464812,16761.41741476904,32101.630885798284,3.820322637472329,0.08830788925796774,0.06083770684635143,null
min,670,03J493690,0000X E 100 ST,0110,ARSON,$300 AND UNDER,ABANDONED BUILDING,111,1,1,0,01A,0,0,2001,36.619446395,-91.686565684,"(36.619446395, -91.686565684)"
max,14274126,ZZ376673,XX S,5132,WEAPONS VIOLATION,WIREROOM/HORSES,YMCA,2535,31,50,77,26,1205119,1951535,2026,42.022671246,-87.524529378,"(42.022671246, -87.677131898)"


df.summary().collect(): 3.70 s


summary,ID,Case Number,Block,IUCR,Primary Type,Description,Location Description,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Latitude,Longitude,Location
count,2214609,2214609,2214609,2214609,2214609,2214609,2202515,2214609,2214608,2196737,2196632,2214609,2117542,2117542,2214609,2117542,2117542,2117542
mean,1.1291700307641665E7,null,null,1086.5438042985438,null,null,null,1148.8881472982364,11.237249210695527,23.159755127718977,36.843304659132706,11.134632881263046,1164941.9766181733,1886275.485424138,2018.0816663347796,41.84352838029658,-87.67022922969481,null
stddev,1778633.200589106,null,null,820.9559413709741,null,null,null,698.7379273929268,6.97672373548794,13.984877062000939,21.490912193034593,6.788762659464812,16761.41741476904,32101.630885798284,3.820322637472329,0.08830788925796774,0.06083770684635143,null
min,670,03J493690,0000X E 100 ST,0110,ARSON,$300 AND UNDER,ABANDONED BUILDING,111,1,1,0,01A,0,0,2001,36.619446395,-91.686565684,"(36.619446395, -91.686565684)"
25%,10740606,null,null,560.0,null,null,null,611,6,10,23,6.0,1153284,1858960,2016,41.768381311,-87.712564504,null
50%,11518159,null,null,860.0,null,null,null,1031,10,24,32,7.0,1166729,1892859,2018,41.861772429,-87.663836332,null
75%,12342205,null,null,1310.0,null,null,null,1722,17,34,54,14.0,1176576,1909015,2021,41.906196882,-87.627688509,null
max,14274126,ZZ376673,XX S,5132,WEAPONS VIOLATION,WIREROOM/HORSES,YMCA,2535,31,50,77,26,1205119,1951535,2026,42.022671246,-87.524529378,"(42.022671246, -87.677131898)"


Temps describe() : 2.939675696999984 secondes
Temps summary()  : 3.7000348259998646 secondes
Nombre de tâches : ouvrir 'See performance' sous cette cellule.


# 12. Valeurs manquantes, valeurs uniques et répartition des crimes

Les chaînes vides sont comptées comme manquantes en plus des valeurs `null`.

Pour le nombre de valeurs uniques :

- le mode par défaut utilise `approx_count_distinct`, plus adapté au Big Data ;
- passer `RUN_EXACT_UNIQUES=true` pour utiliser `countDistinct`, plus coûteux.

Le terme « crime commis » est interprété ici comme **incident enregistré dans le jeu de
données**. Le jeu officiel contient des faits signalés et ses classifications peuvent être
révisées.

In [0]:
missing_expressions = [
    F.sum(
        F.when(
            F.col(column).isNull()
            | (F.trim(F.col(column).cast("string")) == ""),
            F.lit(1),
        ).otherwise(F.lit(0))
    ).cast("long").alias(column)
    for column in df_cached.columns
]

missing_row, missing_seconds = elapsed_call(
    "Comptage des valeurs manquantes",
    lambda: df_cached.agg(*missing_expressions).first(),
)

if RUN_EXACT_UNIQUES:
    unique_expressions = [
        F.countDistinct(F.col(column)).alias(column)
        for column in df_cached.columns
    ]
    unique_label = "Nombre exact de valeurs uniques"
else:
    unique_expressions = [
        F.approx_count_distinct(F.col(column), rsd=0.02).alias(column)
        for column in df_cached.columns
    ]
    unique_label = "Nombre approximatif de valeurs uniques"

unique_row, unique_seconds = elapsed_call(
    unique_label,
    lambda: df_cached.agg(*unique_expressions).first(),
)

quality_rows = [
    (
        column,
        int(missing_row[column] or 0),
        int(unique_row[column] or 0),
        "exact" if RUN_EXACT_UNIQUES else "approximatif",
    )
    for column in df_cached.columns
]

quality_df = spark.createDataFrame(
    quality_rows,
    ["colonne", "valeurs_manquantes", "valeurs_uniques", "mode_uniques"],
)

display(quality_df.orderBy(F.desc("valeurs_manquantes")))


Comptage des valeurs manquantes: 4.63 s
Nombre exact de valeurs uniques: 9.40 s


colonne,valeurs_manquantes,valeurs_uniques,mode_uniques
X Coordinate,97067,70174,exact
Y Coordinate,97067,116702,exact
Latitude,97067,429943,exact
Longitude,97067,429794,exact
Location,97067,430244,exact
Community Area,17977,78,exact
Ward,17872,50,exact
Location Description,12094,175,exact
District,1,23,exact
ID,0,2214609,exact


In [0]:
# Nb tot d'incidents
total_incidents = df_cached.count()

# Effectif et pourcentage / type de crime
crime_counts = (
    df_cached
    .groupBy("Primary Type")
    .agg(
        F.count("*").alias("effectif")
    )
    .withColumn(
        "pourcentage",
        F.round(
            F.col("effectif") / F.lit(total_incidents) * 100,
            4
        )
    )
    .orderBy(
        F.desc("effectif")
    )
)

print(
    "Nombre total d'incidents enregistrés :",
    total_incidents
)

display(crime_counts)

Nombre total d'incidents enregistrés : 2214609


Primary Type,effectif,pourcentage
THEFT,497941,22.4844
BATTERY,405803,18.3239
CRIMINAL DAMAGE,243103,10.9772
ASSAULT,172000,7.7666
DECEPTIVE PRACTICE,166304,7.5094
OTHER OFFENSE,139757,6.3107
MOTOR VEHICLE THEFT,114281,5.1603
NARCOTICS,106775,4.8214
BURGLARY,91938,4.1514
ROBBERY,83240,3.7587


In [0]:
# Vérification : proche de 100 %

percentage_check = (
    crime_counts
    .agg(
        F.sum("effectif").alias("somme_effectifs"),
        F.sum("pourcentage").alias("somme_pourcentages_arrondis")
    )
    .first()
)

print(percentage_check)

Row(somme_effectifs=2214609, somme_pourcentages_arrondis=100.00009999999996)


# 13. Écriture du DataFrame dans HDFS

Cette cellule nécessite qu’un NameNode HDFS soit accessible depuis le cluster Databricks,
par exemple à l’adresse configurée dans `HDFS_TARGET`.

Un espace Databricks DBFS ou Unity Catalog n’est **pas** automatiquement le même système
qu’un cluster HDFS externe. Le format Parquet est retenu, car il conserve les types et est
plus efficace qu’un CSV pour les traitements Big Data.

In [0]:
if RUN_HDFS_WRITE:
    _, hdfs_write_seconds = elapsed_call(
        "Écriture Parquet dans HDFS",
        lambda: (
            df_cached.write
            .mode("overwrite")
            .partitionBy("Year")
            .parquet(HDFS_TARGET)
        ),
    )

    hdfs_df = spark.read.parquet(HDFS_TARGET)
    hdfs_count, hdfs_read_seconds = elapsed_call(
        "Vérification HDFS",
        hdfs_df.count,
    )

    print("Nombre de lignes relues depuis HDFS :", hdfs_count)
    display(hdfs_df.limit(10))
else:
    print(
        "Écriture HDFS désactivée. "
        "Elle nécessite un NameNode accessible depuis Databricks."
    )


PARQUET_PATH = f"{BASE_PATH}/chicago_crimes_parquet"

_, parquet_write_seconds = elapsed_call(
    "Écriture Parquet dans le volume Unity Catalog",
    lambda: (
        df_cached.write
        .mode("overwrite")
        .partitionBy("Year")
        .parquet(PARQUET_PATH)
    ),
)

parquet_df = spark.read.parquet(PARQUET_PATH)
parquet_count = parquet_df.count()

print("Chemin Parquet :", PARQUET_PATH)
print("Nombre de lignes relues :", parquet_count)
display(parquet_df.limit(10))


Écriture HDFS désactivée. Elle nécessite un NameNode accessible depuis Databricks.
Écriture Parquet dans le volume Unity Catalog: 11.43 s
Chemin Parquet : /Volumes/workspace/default/tp-chicagoo/chicago_crimes_parquet
Nombre de lignes relues : 2214609


ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Updated On,Latitude,Longitude,Location,Year
11878504,JC493513,2019-10-30T12:00:00.000Z,023XX E 70TH ST,0820,THEFT,$500 AND UNDER,RESIDENCE,false,true,331,3,5,43,06,1192668,1858995,2019-11-06T15:41:57.000Z,41.768053131,-87.569333606,"(41.768053131, -87.569333606)",2019
11878503,JC494092,2019-11-01T03:08:00.000Z,012XX N CLARK ST,0460,BATTERY,SIMPLE,CTA PLATFORM,false,false,1821,18,2,8,08B,1175314,1908524,2019-11-08T15:42:03.000Z,41.904370109,-87.631460067,"(41.904370109, -87.631460067)",2019
11878499,JC493395,2019-10-31T13:00:00.000Z,009XX W NORTH AVE,0810,THEFT,OVER $500,PARKING LOT/GARAGE(NON.RESID.),false,false,1822,18,27,8,06,1169705,1910843,2019-11-07T15:42:24.000Z,41.910857665,-87.651995629,"(41.910857665, -87.651995629)",2019
11878497,JC494119,2019-11-01T02:11:00.000Z,022XX N CALIFORNIA AVE,2250,LIQUOR LAW VIOLATION,LIQUOR LICENSE VIOLATION,BAR OR TAVERN,true,false,1414,14,1,22,22,1157336,1914799,2019-11-08T15:42:03.000Z,41.921973868,-87.697327162,"(41.921973868, -87.697327162)",2019
11878496,JC494097,2019-11-01T03:14:00.000Z,005XX E 80TH ST,0470,PUBLIC PEACE VIOLATION,RECKLESS CONDUCT,APARTMENT,true,false,624,6,6,44,24,1181163,1852079,2019-11-08T15:42:03.000Z,41.74934759,-87.611716915,"(41.74934759, -87.611716915)",2019
11878495,JC494070,2019-11-01T01:56:00.000Z,002XX N PINE AVE,0520,ASSAULT,AGGRAVATED:KNIFE/CUTTING INSTR,APARTMENT,true,true,1523,15,37,25,04A,1139460,1901108,2019-11-08T15:42:03.000Z,41.88474917,-87.763343624,"(41.88474917, -87.763343624)",2019
11878494,JC494121,2019-11-01T04:50:00.000Z,014XX W CULLERTON ST,1210,DECEPTIVE PRACTICE,THEFT OF LABOR/SERVICES,TAXICAB,false,false,1235,12,25,31,11,1167011,1890515,2019-11-08T15:42:03.000Z,41.855134389,-87.662476146,"(41.855134389, -87.662476146)",2019
11878492,JC494060,2019-10-31T04:50:00.000Z,002XX W ONTARIO ST,0610,BURGLARY,FORCIBLE ENTRY,BAR OR TAVERN,false,false,1831,18,42,8,05,1174477,1904439,2019-11-07T15:42:24.000Z,41.893179406,-87.634656764,"(41.893179406, -87.634656764)",2019
11878491,JC494032,2019-11-01T00:51:00.000Z,029XX W 63RD ST,031A,ROBBERY,ARMED: HANDGUN,RESTAURANT,false,false,823,8,16,66,03,1157658,1862719,2019-11-08T15:42:03.000Z,41.779054042,-87.697560782,"(41.779054042, -87.697560782)",2019
11878490,JC494034,2019-11-01T00:41:00.000Z,0000X W ILLINOIS ST,1310,CRIMINAL DAMAGE,TO PROPERTY,SIDEWALK,false,false,1831,18,42,8,14,1175717,1903591,2019-11-08T15:42:03.000Z,41.890824651,-87.630128275,"(41.890824651, -87.630128275)",2019


# 14. Chargement depuis l’API Socrata

Le compute actuel ne dispose pas d'un accès Internet fonctionnel. La cellule conserve le code
demandé, mais il faut laisser `RUN_API_DOWNLOAD = False` tant que l'accès réseau n'est pas autorisé.


In [0]:
if RUN_API_DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/resource/ijzp-q8t2.json"

    response = requests.get(
        API_URL,
        params={
            "$limit": 50_000,
            "$where": f"year={API_YEAR}",
            "$order": "id",
        },
        timeout=300,
    )
    response.raise_for_status()
    records = response.json()

    print("Nombre de lignes récupérées :", len(records))

    if records:
        api_df = spark.createDataFrame(records)
        display(api_df.limit(20))
else:
    print(
        "Téléchargement API désactivé : "
        "le compute Serverless ne dispose pas actuellement d'un accès Internet."
    )


Téléchargement API désactivé : le compute Serverless ne dispose pas actuellement d'un accès Internet.


In [0]:
print(
    "La lecture API n'est exécutée que lorsque RUN_API_DOWNLOAD = True "
    "et que l'accès Internet est disponible."
)


La lecture API n'est exécutée que lorsque RUN_API_DOWNLOAD = True et que l'accès Internet est disponible.


# 15. Synthèse générale

1. Le fichier utilisé est la version `trimmed` déjà importée dans Unity Catalog.
2. Pandas est pratique, mais charge les données dans la mémoire d'une seule machine.
3. Spark répartit les traitements entre plusieurs partitions.
4. `header=True` récupère les vrais noms des colonnes.
5. `inferSchema=True` est coûteux, car Spark doit analyser les valeurs.
6. Le schéma manuel évite cette passe d'inférence.
7. `display(df)` est l'affichage recommandé dans Databricks.
8. Sur Serverless, les API RDD et certaines API bas niveau ne sont pas disponibles.
9. `collect()` doit être évité sur l'ensemble d'un gros DataFrame.
10. Le cache améliore les traitements répétés.
11. `summary()` fournit davantage d'indicateurs que `describe()`.
12. Les valeurs uniques approximatives sont plus rapides sur les gros volumes.
13. L'écriture Parquet dans Unity Catalog est exécutable sans HDFS externe.
14. L'API nécessite un compute autorisé à accéder à Internet.


# Nettoyage facultatif

La cellule suivante libère le cache Spark. Elle peut être exécutée à la fin du TP.

In [0]:


gc.collect()

print(
    "Nettoyage terminé. "
    "Aucun cache Spark à libérer sur ce compute Serverless."
)

Nettoyage terminé. Aucun cache Spark à libérer sur ce compute Serverless.


# Sources officielles

- Ville de Chicago — Crimes, 2001 à aujourd’hui :  
  `https://data.cityofchicago.org/Public-Safety/Crimes-2001-to-Present/ijzp-q8t2`
- Métadonnées du jeu de données :  
  `https://data.cityofchicago.org/api/views/ijzp-q8t2`
- Export CSV :  
  `https://data.cityofchicago.org/api/views/ijzp-q8t2/rows.csv?accessType=DOWNLOAD`
- API JSON Socrata :  
  `https://data.cityofchicago.org/resource/ijzp-q8t2.json`
- Documentation Spark CSV :  
  `https://spark.apache.org/docs/latest/sql-data-sources-csv.html`
- Documentation PySpark DataFrame :  
  `https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html`
- Documentation Databricks `display()` :  
  `https://docs.databricks.com/aws/en/getting-started/import-visualize-data`
- Pagination Socrata :  
  `https://dev.socrata.com/docs/paging.html`